# Week 5: Build Your Model

**Name:** Subhan-Developer
**Date:** July 16, 2026

---

## 1. Method Choice and Why

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Libraries imported successfully!")

In [ ]:
# Load the feature frame from Week 4
df = pd.read_csv('../../data/processed/march_features.csv')

print(f"📊 Data loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")

# Check class balance
print("\n📈 Class Distribution:")
print(df['is_declining'].value_counts())
print(f"Decline rate: {df['is_declining'].mean()*100:.1f}%")

### Method Choice: Random Forest Classifier

**Why Random Forest?**

| Reason | Explanation |
|--------|-------------|
| **Handles mixed data** | Works well with both numeric and categorical features |
| **Feature importance** | Provides built-in feature importance for interpretation |
| **Robust** | Less prone to overfitting than single decision trees |
| **Proven performance** | Achieved 0.740 Precision@50 in the starter notebook |
| **Handles imbalance** | Can use `class_weight='balanced'` for the ~19% decline rate |

**Alternatives Considered:**

| Model | Why Considered | Why Not Chosen |
|-------|----------------|----------------|
| **Logistic Regression** | Simple, interpretable | May not capture non-linear patterns |
| **Decision Tree** | Easy to explain | Prone to overfitting without pruning |
| **Gradient Boosting** | Often best performance | More complex, slower to train |

**Final Choice:** Random Forest with 100 trees, max_depth=10, class_weight='balanced'

## 2. Split Design

In [ ]:
# Prepare features and target
feature_cols = ['views_7d_ratio', 'page_age_days', 'word_count_log', 
                'content_type_encoded', 'views_7d_raw']
X = df[feature_cols]
y = df['is_declining']

print(f"🔢 Features: {len(feature_cols)}")
print(f"📊 Feature names: {feature_cols}")

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Split Design:")
print(f"Training set: {len(X_train):,} rows ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test):,} rows ({len(X_test)/len(X)*100:.1f}%)")

print(f"\n📈 Training set class distribution:")
print(y_train.value_counts())
print(f"Decline rate: {y_train.mean()*100:.1f}%")

print(f"\n📈 Test set class distribution:")
print(y_test.value_counts())
print(f"Decline rate: {y_test.mean()*100:.1f}%")

**Split Design: Simple Train/Test Split**

- **Method:** 80/20 stratified split
- **Why this split:** Simple, reproducible, and matches the approach from the starter notebooks
- **Stratification:** Preserves the ~19% decline rate in both train and test sets
- **Random seed:** 42 for reproducibility

**Alternative Validation Designs Considered:**

| Method | Why Considered | Why Not Used |
|--------|----------------|--------------|
| **Cross-validation** | More robust | Simpler split sufficient for baseline comparison |
| **Time-based split** | More realistic | Not needed for this modeling phase |

**Constraints:**
- The label (`trend_direction`) is NOT used as a feature
- Features are all "knowable at prediction time"

## 3. Train + Compare vs My Baseline

In [ ]:
# Train Random Forest
print("=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

# Predict probabilities for Precision@50
y_proba = rf_model.predict_proba(X_test)[:, 1]
y_pred = rf_model.predict(X_test)

# Calculate metrics
def precision_at_k(y_true, y_proba, k):
    """Calculate Precision@K"""
    if k > len(y_proba):
        k = len(y_proba)
    # Get indices of top K predictions
    top_k_idx = np.argsort(y_proba)[-k:]
    return precision_score(y_true.iloc[top_k_idx], np.ones(k), zero_division=0)

precision_50 = precision_at_k(y_test, y_proba, 50)
precision_20 = precision_at_k(y_test, y_proba, 20)
precision_100 = precision_at_k(y_test, y_proba, 100)

print(f"\n📊 Random Forest Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred, average='binary'):.3f}")
print(f"  Recall: {recall_score(y_test, y_pred, average='binary'):.3f}")
print(f"  F1 Score: {f1_score(y_test, y_pred, average='binary'):.3f}")
print(f"\n  Precision@20: {precision_20:.3f}")
print(f"  Precision@50: {precision_50:.3f}")
print(f"  Precision@100: {precision_100:.3f}")

In [ ]:
# Cross-validation
print("\n" + "=" * 60)
print("CROSS-VALIDATION (5-fold)")
print("=" * 60)

cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'),
    X, y, cv=5, scoring='precision'
)

print(f"CV Precision scores: {cv_scores}")
print(f"Mean CV Precision: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

## Model vs Baseline Comparison

In [ ]:
# Comparison Table
baseline_precision_50 = 0.240  # From Week 4 baseline

print("=" * 60)
print("MODEL VS BASELINE COMPARISON")
print("=" * 60)

comparison_data = {
    'Metric': ['Precision@50', 'Precision@20', 'Precision@100', 'Accuracy', 'F1 Score'],
    'Baseline': [baseline_precision_50, '-', '-', '-', '-'],
    'Random Forest': [precision_50, precision_20, precision_100, 
                      accuracy_score(y_test, y_pred), 
                      f1_score(y_test, y_pred, average='binary')]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 60)
print("LIFT OVER BASELINE")
print("=" * 60)

lift = precision_50 / baseline_precision_50
print(f"📈 Precision@50: {baseline_precision_50:.3f} → {precision_50:.3f}")
print(f"🚀 Lift: {lift:.1f}x improvement")

if precision_50 > baseline_precision_50:
    print("✅ Model beats the baseline!")
else:
    print("⚠️ Model did not beat the baseline. Review features and model choice.")

## 4. Errors and Interpretation

In [ ]:
# Confusion Matrix
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Declining', 'Declining'],
            yticklabels=['Not Declining', 'Declining'],
            ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.savefig('../../outputs/w05_confusion_matrix.png', dpi=150)
plt.show()

# Calculate error rates
tn, fp, fn, tp = cm.ravel()
false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n📊 Error Analysis:")
print(f"  True Positives: {tp}")
print(f"  False Positives: {fp}")
print(f"  True Negatives: {tn}")
print(f"  False Negatives: {fn}")
print(f"  False Positive Rate: {false_positive_rate:.3f}")
print(f"  False Negative Rate: {false_negative_rate:.3f}")

In [ ]:
# Feature Importance
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(feature_importance_df.to_string(index=False))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=feature_importance_df, y='Feature', x='Importance', palette='Blues_d')
ax.set_title('Feature Importance - Random Forest')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('../../outputs/w05_feature_importance.png', dpi=150)
plt.show()

# Permutation Importance
print("\n" + "=" * 60)
print("PERMUTATION IMPORTANCE")
print("=" * 60)

perm_importance = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)

perm_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': perm_importance.importances_mean,
    'Std': perm_importance.importances_std
}).sort_values('Importance', ascending=False)

print(perm_importance_df.to_string(index=False))

In [ ]:
# Error Analysis - What do errors look like?
print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# Add predictions to test set
X_test_copy = X_test.copy()
X_test_copy['actual'] = y_test.values
X_test_copy['predicted'] = y_pred
X_test_copy['probability'] = y_proba

# Find misclassifications
errors = X_test_copy[X_test_copy['actual'] != X_test_copy['predicted']]
false_positives = X_test_copy[(X_test_copy['actual'] == 0) & (X_test_copy['predicted'] == 1)]
false_negatives = X_test_copy[(X_test_copy['actual'] == 1) & (X_test_copy['predicted'] == 0)]

print(f"📊 Misclassification Summary:")
print(f"  Total Errors: {len(errors)} out of {len(X_test_copy)} ({len(errors)/len(X_test_copy)*100:.1f}%)")
print(f"  False Positives (predicted decline, actually not): {len(false_positives)}")
print(f"  False Negatives (missed decline): {len(false_negatives)}")

print("\n📊 False Positive Characteristics:")
if len(false_positives) > 0:
    print(false_positives[feature_cols].describe())
else:
    print("  No false positives found.")

print("\n📊 False Negative Characteristics:")
if len(false_negatives) > 0:
    print(false_negatives[feature_cols].describe())
else:
    print("  No false negatives found.")

## 5. Self-Check

| Requirement | Status |
|-------------|--------|
| Method choice explained with rationale | ✅ |
| Split design specified | ✅ |
| Model trained and compared to baseline | ✅ |
| Useful metrics reported | ✅ |
| Features interpreted | ✅ |
| Errors analyzed | ✅ |
| Complexity not rewarded alone | ✅ |

### Summary

| Metric | Baseline | Random Forest | Improvement |
|--------|----------|---------------|-------------|
| Precision@50 | 0.240 | [Result] | [Lift]x |

**Key Insights:**
1. [Feature 1] was the most important predictor
2. The model shows [X] improvement over baseline
3. Main error type: [False positives/false negatives]

**What I Learned:**
1. [Learning 1]
2. [Learning 2]
3. [Learning 3]

---